In [ ]:
import requests
import pandas as pd
import time #for optional rate limiting
import sys
print(sys.version)

In [ ]:
# Base URL for the API
url = "https://clinicaltrials.gov/api/v2/studies"

# Storage for all trials
all_studies = []

# Initialize pagination 
next_page_token = None

# Set query term
query_term = "diabetes"

#Max per page
page_size = 1000

In [ ]:
# Loop to handle pagination
while True:
    params = {
        "query.term": "diabetes",
        "pageSize": 1000, #max safe chunk
    }
    if next_page_token:
        params["pageToken"] = next_page_token

    response = requests.get(url, params=params)
    
    #Check for successful response
    if response.status_code != 200:
        print(f"Error fetching data: {response.status_code}")
        print(response.text[:500])
        break

    # Parse JSON response 
    try:
        data = response.json()
    except ValueError as e:
        print(f"Error parsing JSON: {e}")
        print(response.text[:500])
        break

    studies = data.get("studies", [])
    all_studies.extend(studies)

    print(f"Pulled {len(all_studies)} trials so far...")

    next_page_token = data.get("nextPageToken")
    if not next_page_token:
        break

print(f"Final total trials pulled: {len(all_studies)}")

In [ ]:
print(len(all_studies))

In [ ]:
import json
print(json.dumps(all_studies[0] ["protocolSection"], indent=2))

In [ ]:
#Extract relevant fields
records = []

for study in all_studies:
    record = {
        "nct_id": study.get("protocolSection", {})
                        .get("identificationModule", {})
                        .get("nctId"),

        "title": study.get("protocolSection", {})
                      .get("identificationModule", {})
                      .get("officialTitle"),

        "status": study.get("protocolSection", {})
                       .get("statusModule", {})
                       .get("overallStatus"),

        "study_type": study.get("protocolSection", {})
                           .get("designModule", {})
                           .get("studyType"),
        
        "intervention_type": study.get("protocolSection", {})
                                    .get("armsInterventionsModule", {})
                                    .get("interventions"),

        "phase": (
            study.get("protocolSection", {})
                 .get("designModule", {})
                 .get("phases")
            or study.get("protocolSection", {})
                    .get("designModule", {})
                    .get("phaseList", {})
                    .get("phases")
            or study.get("protocolSection", {})
                    .get("designModule", {})
                    .get("phase")
            or "NA"
        ),
        
        "locations": len(
            study.get("protocolSection", {})
                 .get("contactsLocationsModule", {})
                 .get("locations",[])
    ),

        "start_date": study.get("protocolSection", {})
                           .get("statusModule", {})
                           .get("startDateStruct", {})
                           .get("date"),

        "completion_date": study.get("protocolSection", {})
                                .get("statusModule", {})
                                .get("completionDateStruct", {})
                                .get("date"),

        "enrollment": study.get("protocolSection", {})
                           .get("designModule", {})
                           .get("enrollmentInfo", {})
                           .get("count"),
    }

    records.append(record)

df = pd.DataFrame(records)
print(df.shape)

In [ ]:
import json

print(json.dumps(
    all_studies[0]["protocolSection"]
    .get("contactsLocationsModule", {}),
    indent=2
))

In [ ]:
df[['nct_id', 'enrollment', 'study_type']].head()

In [ ]:
df['locations'].describe()
df['locations'].value_counts().head(10)

In [ ]:
df = df[df['study_type'] == "INTERVENTIONAL"]
# Randomly sample 5000 trials for analysis
df = df.sample(n=5000, random_state=42)

print(df.shape)

In [ ]:
import os
print(os.getcwd())

In [ ]:
# Save raw data
import os
os.makedirs("../data/raw/", exist_ok=True)
df.to_csv("../data/raw/interventional_diabetes_trials_raw.csv", index=False)